# Minimal Trimmed Mean PCE

This notebook keeps only the steps needed to write:

- `outputs/trimmed_mean_rates.csv`
- `outputs/component_monthly_audit.csv`
- `outputs/trimmed_away_components.csv`

It assumes `Trimmed_Mean_Components.xlsx` has the component line items in column A.

In [9]:
import json
import time
import urllib.parse
import urllib.request
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

BEA_API_KEY = "FD4E7F46-5D89-4CA8-BE48-4F5997F83B3B"

BASE_DIR = Path.cwd()
COMPONENT_FILE = BASE_DIR / "Trimmed_Mean_Components.xlsx"
OUTPUT_DIR = BASE_DIR / "outputs"

# First month to include in the output rates and audit tables.
START_DATE = pd.Timestamp("1980-01-01")

LOWER_TRIM = 0.0 #0.24
UPPER_TRIM = 0.0 #0.31
RETAINED_MASS = 1.0 - LOWER_TRIM - UPPER_TRIM

BEA_URL = "https://apps.bea.gov/api/data"
BEA_TABLES = {
    "prices": "U20404",
    "nominal": "U20405",
    "real": "U20406",
}

## 1. Load Components

In [2]:
components = pd.read_excel(COMPONENT_FILE, usecols=[0, 1, 2])
components.columns = ["line_item", "description", "code"]
components = components.dropna(subset=["line_item"]).copy()
components["line_item"] = components["line_item"].astype(int)
components["description"] = components["description"].astype(str).str.strip()
components["code"] = components["code"].astype(str).str.strip()
components = components.sort_values("line_item").reset_index(drop=True)

component_lines = components["line_item"].tolist()

# BEA combines workbook lines 157 and 158 into line 156 after 2001.
# Pull line 156 only so those two workbook lines can be filled consistently.
bea_lines = sorted(set(component_lines + [156]))
components.head()

,line_item,description,code
0,7,New domestic autos,DNDCRG
1,8,New foreign autos,DNFCRG
2,9,New light trucks,DNWTRG
3,13,Used autos,DNPURG
4,17,Used light trucks,DUTRRG


## 2. Pull BEA Tables

The API key is passed directly in the request parameters.

In [3]:
pull_start_year = (START_DATE - pd.DateOffset(months=1)).year
end_year = date.today().year

raw_tables = {}

for label, table_name in BEA_TABLES.items():
    rows = []
    skipped_years = []

    for year in range(pull_start_year, end_year + 1):
        params = {
            "UserID": BEA_API_KEY,
            "method": "GetData",
            "DataSetName": "NIUnderlyingDetail",
            "TableName": table_name,
            "Frequency": "M",
            "Year": str(year),
            "ResultFormat": "JSON",
        }
        url = f"{BEA_URL}?{urllib.parse.urlencode(params)}"

        with urllib.request.urlopen(url, timeout=90) as response:
            payload = json.loads(response.read().decode("utf-8-sig"))

        error = payload.get("BEAAPI", {}).get("Error")
        if error:
            skipped_years.append(year)
            continue

        data = payload["BEAAPI"]["Results"].get("Data", [])
        if isinstance(data, dict):
            data = [data]
        rows.extend(data)
        time.sleep(0.05)

    raw_tables[label] = pd.DataFrame(rows)

    note = ""
    if skipped_years:
        note = f"; skipped years without monthly data: {min(skipped_years)}-{max(skipped_years)}"
    print(f"{table_name} ({label}): {len(raw_tables[label]):,} rows{note}")

U20404 (prices): 220,232 rows
U20405 (nominal): 221,368 rows
U20406 (real): 92,800 rows; skipped years without monthly data: 1979-2006


## 3. Convert BEA Rows To Monthly Matrices

In [10]:
matrices = {}

for label, raw in raw_tables.items():
    df = raw.copy()
    df["line_item"] = pd.to_numeric(df["LineNumber"], errors="coerce").astype("Int64")
    df["date"] = pd.to_datetime(
        df["TimePeriod"].astype(str).str.replace(
            r"^(\d{4})M(\d{1,2})$", r"\1-\2-01", regex=True
        )
    )
    df["value"] = pd.to_numeric(
        df["DataValue"].astype(str).str.replace(",", "", regex=False),
        errors="coerce",
    )
    df = df[df["line_item"].isin(bea_lines)]

    matrix = (
        df.pivot_table(index="date", columns="line_item", values="value", aggfunc="first")
        .sort_index()
        .reindex(columns=bea_lines)
    )
    matrix.columns = matrix.columns.astype(int)
    matrices[label] = matrix

prices = matrices["prices"]
nominal = matrices["nominal"]
real = matrices["real"]

## 4. Fill The Tenant-Housing Line Break

Workbook lines 157 and 158 stop appearing separately in BEA after 2001.
From 2002 onward, this fills those two lines from BEA aggregate line 156,
split using their December 2001 nominal-expenditure shares.

In [11]:
split_start = pd.Timestamp("2002-01-01")
aggregate_line = 156
child_lines = [157, 158]

split_values = nominal.loc[nominal.index < split_start, child_lines].dropna().iloc[-1]
split_ratios = split_values / split_values.sum()

for child_line in child_lines:
    fill = prices[child_line].isna() & prices[aggregate_line].notna() & (prices.index >= split_start)
    prices.loc[fill, child_line] = prices.loc[fill, aggregate_line]

    for matrix in [nominal, real]:
        fill = matrix[child_line].isna() & matrix[aggregate_line].notna() & (matrix.index >= split_start)
        matrix.loc[fill, child_line] = matrix.loc[fill, aggregate_line] * split_ratios.loc[child_line]

prices = prices.drop(columns=[aggregate_line], errors="ignore").reindex(columns=component_lines)
nominal = nominal.drop(columns=[aggregate_line], errors="ignore").reindex(columns=component_lines)
real = real.drop(columns=[aggregate_line], errors="ignore").reindex(columns=component_lines)

## 5. Monthly Price Changes And Dallas Fed Weights

In [12]:
calc_start = START_DATE - pd.DateOffset(months=1)

quantity = real.combine_first(nominal / prices)

common_months = prices.index.intersection(quantity.index)
prices = prices.loc[common_months].sort_index()
quantity = quantity.loc[common_months].sort_index()

prices = prices.loc[prices.index >= calc_start]
quantity = quantity.loc[prices.index]

price_changes = prices.pct_change().iloc[1:]

p_lag = prices.shift(1).iloc[1:].to_numpy(dtype=float)
q_current = quantity.iloc[1:].to_numpy(dtype=float)
q_lag = quantity.shift(1).iloc[1:].to_numpy(dtype=float)

weight_1 = q_current * p_lag
weight_1 = weight_1 / weight_1.sum(axis=1, keepdims=True)

weight_2 = q_lag * p_lag
weight_2 = weight_2 / weight_2.sum(axis=1, keepdims=True)

weights = 0.5 * weight_1 + 0.5 * weight_2
weights = weights / weights.sum(axis=1, keepdims=True)
weights = pd.DataFrame(weights, index=price_changes.index, columns=prices.columns)

price_changes = price_changes.loc[price_changes.index >= START_DATE]
weights = weights.loc[price_changes.index]

## 6. Weighted Trim

In [13]:
def trim_one_month(changes, weights):
    month = pd.DataFrame(
        {
            "line_item": changes.index.astype(int),
            "price_change": changes.to_numpy(dtype=float),
            "raw_weight": weights.reindex(changes.index).to_numpy(dtype=float),
        }
    ).sort_values(["price_change", "line_item"], kind="mergesort").reset_index(drop=True)

    month["raw_weight"] = month["raw_weight"] / month["raw_weight"].sum()
    month["rank"] = np.arange(1, len(month) + 1)

    cdf_end = month["raw_weight"].cumsum().to_numpy()
    cdf_start = np.r_[0.0, cdf_end[:-1]]

    retain_low = LOWER_TRIM
    retain_high = 1.0 - UPPER_TRIM

    month["cdf_start"] = cdf_start
    month["cdf_end"] = cdf_end
    month["retained_weight"] = np.maximum(
        0.0, np.minimum(cdf_end, retain_high) - np.maximum(cdf_start, retain_low)
    )
    month["lower_trimmed_weight"] = np.maximum(0.0, np.minimum(cdf_end, retain_low) - cdf_start)
    month["upper_trimmed_weight"] = np.maximum(0.0, cdf_end - np.maximum(cdf_start, retain_high))
    month["trimmed_weight"] = month["lower_trimmed_weight"] + month["upper_trimmed_weight"]
    month["included"] = month["retained_weight"] > 1e-14
    month["trimmed_away"] = month["trimmed_weight"] > 1e-14
    month["trim_side"] = np.select(
        [
            (month["lower_trimmed_weight"] > 1e-14) & (month["upper_trimmed_weight"] > 1e-14),
            month["lower_trimmed_weight"] > 1e-14,
            month["upper_trimmed_weight"] > 1e-14,
        ],
        ["both", "lower", "upper"],
        default="none",
    )

    trimmed_change = (month["retained_weight"] * month["price_change"]).sum() / RETAINED_MASS
    return float(trimmed_change), month


trimmed_changes = {}
audit_rows = []

for month in price_changes.index:
    trimmed_change, audit = trim_one_month(price_changes.loc[month], weights.loc[month])
    audit.insert(0, "date", month)
    audit.insert(1, "period", month.strftime("%Y-%m"))
    trimmed_changes[month] = trimmed_change
    audit_rows.append(audit)

trimmed_changes = pd.Series(trimmed_changes, name="trimmed_monthly_change").sort_index()

component_meta = components.set_index("line_item")[["code", "description"]]
component_monthly_audit = pd.concat(audit_rows, ignore_index=True).join(component_meta, on="line_item")
component_monthly_audit = component_monthly_audit[
    [
        "date",
        "period",
        "line_item",
        "code",
        "description",
        "rank",
        "price_change",
        "raw_weight",
        "retained_weight",
        "trimmed_weight",
        "lower_trimmed_weight",
        "upper_trimmed_weight",
        "trim_side",
        "included",
        "trimmed_away",
        "cdf_start",
        "cdf_end",
    ]
]

## 7. Build Rates And Write CSVs

In [14]:
trimmed_index = pd.concat(
    [
        pd.Series([100.0], index=[calc_start]),
        100.0 * (1.0 + trimmed_changes).cumprod(),
    ]
).sort_index()

if pd.Timestamp("2009-01-01") in trimmed_index.index:
    trimmed_index = 100.0 * trimmed_index / trimmed_index.loc[pd.Timestamp("2009-01-01")]

trimmed_mean_rates = pd.DataFrame(index=trimmed_index.index)
trimmed_mean_rates["trimmed_mean_index"] = trimmed_index
trimmed_mean_rates["trimmed_monthly_change"] = trimmed_changes.reindex(trimmed_index.index)
trimmed_mean_rates["trimmed_mean_1m_annualized"] = 100.0 * (
    (trimmed_index / trimmed_index.shift(1)) ** 12 - 1.0
)
trimmed_mean_rates["trimmed_mean_6m_annualized"] = 100.0 * (
    (trimmed_index / trimmed_index.shift(6)) ** 2 - 1.0
)
trimmed_mean_rates["trimmed_mean_12m"] = 100.0 * (trimmed_index / trimmed_index.shift(12) - 1.0)

trimmed_mean_rates = trimmed_mean_rates.loc[START_DATE:].copy()
trimmed_mean_rates.insert(0, "period", trimmed_mean_rates.index.strftime("%Y-%m"))
trimmed_mean_rates.insert(0, "date", trimmed_mean_rates.index.strftime("%Y-%m-%d"))
trimmed_mean_rates = trimmed_mean_rates.reset_index(drop=True)

trimmed_away_components = component_monthly_audit.loc[
    component_monthly_audit["trimmed_away"]
].copy()

# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# trimmed_mean_rates.to_csv(OUTPUT_DIR / "trimmed_mean_rates.csv", index=False)
# component_monthly_audit.to_csv(OUTPUT_DIR / "component_monthly_audit.csv", index=False)
# trimmed_away_components.to_csv(OUTPUT_DIR / "trimmed_away_components.csv", index=False)

# print(f"Wrote CSVs to {OUTPUT_DIR}")
# print(f"Months: {trimmed_mean_rates['period'].iloc[0]} to {trimmed_mean_rates['period'].iloc[-1]}")
trimmed_mean_rates.tail()

,date,period,trimmed_mean_index,trimmed_monthly_change,trimmed_mean_1m_annualized,trimmed_mean_6m_annualized,trimmed_mean_12m
551,2025-12-01,2025-12,146.888558,0.003341,4.083672,3.026573,2.878586
552,2026-01-01,2026-01,147.372911,0.003297,4.029459,3.276433,2.855357
553,2026-02-01,2026-02,147.964552,0.004015,4.925315,3.547086,2.861237
554,2026-03-01,2026-03,148.966069,0.006769,8.431654,4.398575,3.542407
555,2026-04-01,2026-04,149.574276,0.004083,5.010960,4.844103,3.827025
